# Homework 2: training pipeline

This code will test your homework 2 solutions by using them in a complete ML pipeline. You should run this code in order to tune your model and save your model weights (which will also be uploaded as part of your solution)

In [1]:
# Download the training data from the homework2 folder:
# unzip using tar xzvvf nsynth_subset.tar.gz
# (this is a small subset of the "nsynth" dataset: https://magenta.tensorflow.org/datasets/nsynth)

In [2]:
import homework2

### Install and Load Required Libraries  

In [3]:
# !pip install librosa
# !pip install torch
# !pip install glob
# !pip install numpy

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import librosa
import random
import glob

In [5]:
BATCH_SIZE = 16
torch.use_deterministic_algorithms(True)

In [6]:
if not len(homework2.audio_paths):
    print("You probably need to set the dataroot folder correctly")

In [7]:
# Some helper functions. These are the same as what the autograder runs.

In [8]:
# Split dataset into train / valid / test
def split_data(waveforms, labels, train_ratio=0.7, valid_ratio=0.15):
    assert(train_ratio + valid_ratio < 1)
    test_ratio = 1 - (train_ratio + valid_ratio)
    N = len(waveforms)
    Ntrain = int(N * train_ratio)
    Nvalid = int(N * valid_ratio)
    Ntest = int(N * test_ratio)
    Wtrain = waveforms[:Ntrain]
    Wvalid = waveforms[Ntrain:Ntrain + Nvalid]
    Wtest = waveforms[Ntrain + Nvalid:]
    ytrain = labels[:Ntrain]
    yvalid = labels[Ntrain:Ntrain + Nvalid]
    ytest = labels[Ntrain + Nvalid:]
    return Wtrain,Wvalid,Wtest,ytrain,yvalid,ytest

In [9]:
def process_data(W, feature_function):
    return [feature_function(path) for path in W]

In [10]:
class InstrumentDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        features = self.features[idx]
        label = self.labels[idx]

        return features, torch.tensor(label, dtype=torch.long)

In [11]:
class Loaders():
    def __init__(self, waveforms, labels, feature_function, seed = 0):
        torch.manual_seed(seed)
        random.seed(seed)
        self.Wtrain, self.Wvalid, self.Wtest, self.ytrain, self.yvalid, self.ytest = split_data(waveforms, labels)
        
        self.Xtrain = process_data(self.Wtrain, feature_function)
        self.Xvalid = process_data(self.Wvalid, feature_function)
        self.Xtest = process_data(self.Wtest, feature_function)
        
        self.dataTrain = InstrumentDataset(self.Xtrain, self.ytrain)
        self.dataValid = InstrumentDataset(self.Xvalid, self.yvalid)
        self.dataTest = InstrumentDataset(self.Xtest, self.ytest)
        
        self.loaderTrain = DataLoader(self.dataTrain, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        self.loaderValid = DataLoader(self.dataValid, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        self.loaderTest = DataLoader(self.dataTest, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [12]:
class Pipeline():
    def __init__(self, module, learning_rate, seed = 0):
        # These two lines will (mostly) make things deterministic.
        # You're welcome to modify them to try to get a better solution.
        torch.manual_seed(seed)
        random.seed(seed)

        self.device = torch.device("cpu") # Can change this if you have a GPU, but the autograder will use CPU
        self.criterion = nn.CrossEntropyLoss()
        
        self.model = module.to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

    def evaluate(self, loader, which = "valid"):
        self.model.eval()

        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                outputs = self.model(inputs)
                #loss = criterion(outputs, labels) # validation loss

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        acc = correct / total
        
        return acc
    
    def train(self, loaders,
          num_epochs=1, # Train for a single epoch by default
          model_path=None): # (Optionally) provide a path to save the best model
        val_acc = 0
        best_val_acc = 0
        for epoch in range(num_epochs):
            self.model.train()
            
            losses = []

            for inputs, labels in loaders.loaderTrain:
                inputs, labels = inputs.to(self.device), labels.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                losses.append(float(loss))
            
            self.model.eval()
            val_acc = self.evaluate(loaders.loaderValid)
            print("Epoch " + str(epoch) + ", loss = " + str(sum(losses)/len(losses)) +\
                  ", validation accuracy = " + str(val_acc))

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                if (model_path):
                    torch.save(self.model.state_dict(), model_path)
        print("Final validation accuracy = " + str(val_acc) + ", best = " + str(best_val_acc))
        return val_acc, best_val_acc

    def load(self, path):
        self.model.load_state_dict(torch.load(path, weights_only=True))

In [13]:
# The function below is the basis of how the autograder tests your code. Try to understand this one.

In [14]:
def test(waveforms, labels, feature_func, classifier, learning_rate, path):
    print("Extracting features...")
    test_loaders = Loaders(waveforms, labels, feature_func)
    test_pipeline = Pipeline(classifier, learning_rate)
    
    # Note: the autograder will not run this line: it will just load your saved model (next line)
    acc, best_acc = test_pipeline.train(test_loaders, 10, path)
    
    test_pipeline.load(path)
    test_acc = test_pipeline.evaluate(test_loaders.loaderTest)
    print("Test accuracy = " + str(test_acc))

In [15]:
# 1. Paths, labels, waveforms

In [16]:
# Once you've written the corresponding code in homework2.py, print these out or visualize them if you want
homework2.waveforms
homework2.labels

[0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,


In [17]:
# 2. MFCC

In [18]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_mfcc,
     homework2.MLPClassifier(),
     0.0001,
     "best_mlp_model.weights")

Extracting features...
Epoch 0, loss = 3.1391650570763483, validation accuracy = 0.5772357723577236
Epoch 1, loss = 0.8373606842425134, validation accuracy = 0.6666666666666666
Epoch 2, loss = 0.4870854939023654, validation accuracy = 0.8536585365853658
Epoch 3, loss = 0.3605955218275388, validation accuracy = 0.9186991869918699
Epoch 4, loss = 0.28160931169986725, validation accuracy = 0.926829268292683
Epoch 5, loss = 0.23399612804253897, validation accuracy = 0.943089430894309
Epoch 6, loss = 0.19937572607563603, validation accuracy = 0.9512195121951219
Epoch 7, loss = 0.17265672836866644, validation accuracy = 0.9512195121951219
Epoch 8, loss = 0.1515608142233557, validation accuracy = 0.9512195121951219
Epoch 9, loss = 0.1340621088941892, validation accuracy = 0.959349593495935
Final validation accuracy = 0.959349593495935, best = 0.959349593495935
Test accuracy = 0.9435483870967742


In [19]:
# 3. Spectrogram

In [31]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_spec,
     homework2.SimpleCNN(),
     0.0001,
     "best_spec_model.weights")

Extracting features...
Epoch 0, loss = 0.6045351409249835, validation accuracy = 0.9105691056910569
Epoch 1, loss = 0.5399324537979232, validation accuracy = 0.9105691056910569
Epoch 2, loss = 0.51068355060286, validation accuracy = 0.926829268292683
Epoch 3, loss = 0.4859732795092795, validation accuracy = 0.926829268292683
Epoch 4, loss = 0.45881304641564685, validation accuracy = 0.943089430894309
Epoch 5, loss = 0.4272314069999589, validation accuracy = 0.9512195121951219
Epoch 6, loss = 0.3936885885066456, validation accuracy = 0.959349593495935
Epoch 7, loss = 0.3633054147164027, validation accuracy = 0.967479674796748
Epoch 8, loss = 0.33684709916512173, validation accuracy = 0.967479674796748
Epoch 9, loss = 0.3138097744021151, validation accuracy = 0.975609756097561
Epoch 10, loss = 0.2949492066270775, validation accuracy = 0.975609756097561
Epoch 11, loss = 0.2794070922666126, validation accuracy = 0.991869918699187
Epoch 12, loss = 0.2662361210419072, validation accuracy = 0

KeyboardInterrupt: 

In [21]:
# 4. Mel-spectrogram

In [22]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_mel,
     homework2.SimpleCNN(),
     0.0001,
     "best_mel_model.weights")

Extracting features...
Epoch 0, loss = 0.49733755737543106, validation accuracy = 0.8211382113821138
Epoch 1, loss = 0.34611210020052063, validation accuracy = 0.8861788617886179
Epoch 2, loss = 0.2898189557923211, validation accuracy = 0.926829268292683
Epoch 3, loss = 0.25354690063330865, validation accuracy = 0.959349593495935
Epoch 4, loss = 0.22609425687955487, validation accuracy = 0.975609756097561
Epoch 5, loss = 0.20433712357448208, validation accuracy = 0.991869918699187
Epoch 6, loss = 0.18600525127516854, validation accuracy = 1.0
Epoch 7, loss = 0.1697208130111297, validation accuracy = 1.0
Epoch 8, loss = 0.1550865475502279, validation accuracy = 1.0
Epoch 9, loss = 0.14196770990060437, validation accuracy = 1.0
Final validation accuracy = 1.0, best = 1.0
Test accuracy = 0.9596774193548387


In [23]:
# 5. Constant-Q

In [24]:
test(homework2.waveforms,
     homework2.labels,
     homework2.extract_q,
     homework2.SimpleCNN(),
     0.0001,
     "best_q_model.weights")

Extracting features...
Epoch 0, loss = 0.5847889458139738, validation accuracy = 0.7317073170731707
Epoch 1, loss = 0.4942872201402982, validation accuracy = 0.8373983739837398
Epoch 2, loss = 0.4359193435973591, validation accuracy = 0.8699186991869918
Epoch 3, loss = 0.3920159935951233, validation accuracy = 0.8861788617886179
Epoch 4, loss = 0.3559260124133693, validation accuracy = 0.9024390243902439
Epoch 5, loss = 0.32543645923336345, validation accuracy = 0.9349593495934959
Epoch 6, loss = 0.2995281331241131, validation accuracy = 0.9349593495934959
Epoch 7, loss = 0.27700910675856805, validation accuracy = 0.943089430894309
Epoch 8, loss = 0.25685692247417236, validation accuracy = 0.9512195121951219
Epoch 9, loss = 0.23882122172249687, validation accuracy = 0.959349593495935
Final validation accuracy = 0.959349593495935, best = 0.959349593495935
Test accuracy = 0.9516129032258065


In [25]:
# 6. Pitch shift

In [26]:
test(homework2.augmented_waveforms,
     homework2.augmented_labels,
     homework2.extract_q,
     homework2.SimpleCNN(),
     0.0001,
     "best_augmented_model.weights")

Extracting features...
Epoch 0, loss = 0.5398882521247422, validation accuracy = 0.8834688346883469
Epoch 1, loss = 0.4076905338852494, validation accuracy = 0.9349593495934959
Epoch 2, loss = 0.3508066398402055, validation accuracy = 0.948509485094851
Epoch 3, loss = 0.3158304425025428, validation accuracy = 0.967479674796748
Epoch 4, loss = 0.28910264904024424, validation accuracy = 0.978319783197832
Epoch 5, loss = 0.2659969516788368, validation accuracy = 0.978319783197832
Epoch 6, loss = 0.24524720351177234, validation accuracy = 0.981029810298103
Epoch 7, loss = 0.22643355763068906, validation accuracy = 0.986449864498645
Epoch 8, loss = 0.20910351406092997, validation accuracy = 0.986449864498645
Epoch 9, loss = 0.19282441689736313, validation accuracy = 0.991869918699187
Final validation accuracy = 0.991869918699187, best = 0.991869918699187
Test accuracy = 0.9972972972972973


In [27]:
# 7. Extend your model to handle four classes and creatively improve its performance

In [28]:
homework2.feature_func_7 = homework2.extract_mel

In [29]:
def test(waveforms, labels, feature_func, classifier, learning_rate, path):
    print("Extracting features...")
    test_loaders = Loaders(waveforms, labels, feature_func)
    test_pipeline = Pipeline(classifier, learning_rate)
    
    # Note: the autograder will not run this line: it will just load your saved model (next line)
    acc, best_acc = test_pipeline.train(test_loaders, 30, path)
    
    test_pipeline.load(path)
    test_acc = test_pipeline.evaluate(test_loaders.loaderTest)
    print("Test accuracy = " + str(test_acc))

In [30]:
test(homework2.waveforms,
     homework2.labels_7,
     homework2.feature_func_7,
     homework2.model_7,
     0.001,
     "best_model_7.weights")

Extracting features...
Epoch 0, loss = 0.632258739736345, validation accuracy = 0.4796747967479675
Epoch 1, loss = 0.36079255160358215, validation accuracy = 0.7967479674796748
Epoch 2, loss = 0.2818380751543575, validation accuracy = 0.8617886178861789
Epoch 3, loss = 0.23491031966275638, validation accuracy = 0.8373983739837398
Epoch 4, loss = 0.1983321662992239, validation accuracy = 0.8943089430894309
Epoch 5, loss = 0.17330563316742578, validation accuracy = 0.8536585365853658
Epoch 6, loss = 0.1576424033070604, validation accuracy = 0.943089430894309
Epoch 7, loss = 0.1396998084253735, validation accuracy = 0.9186991869918699
Epoch 8, loss = 0.11891780690186554, validation accuracy = 0.9186991869918699
Epoch 9, loss = 0.101381563840227, validation accuracy = 0.975609756097561
Epoch 10, loss = 0.09201228308180968, validation accuracy = 0.8861788617886179
Epoch 11, loss = 0.08882690844539967, validation accuracy = 0.8780487804878049
Epoch 12, loss = 0.08138828905713227, validation 